# Практикум 8. Разметка последовательностей: скрытые марковские модели (HMM) и алгоритм Витерби

В лаборатории 8 вы заполняли решётку Витерби шаг за шагом. Здесь сделаем то же самое в коде:

1. по маленькому размеченному корпусу (теги UD UPOS) оценим вероятности переходов и эмиссий методом максимального правдоподобия со сглаживанием;
2. напишем алгоритм Витерби с нуля и распечатаем решётку по шагам;
3. проверим Витерби полным перебором и сравним с жадным декодированием;
4. измерим точность на отложенных предложениях и научим модель угадывать теги незнакомых слов по окончанию;
5. сравним нашу модель с HMM из `nltk`.

*Практикум к лаборатории 8 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Корпус и параметры HMM

Разметка последовательностей (sequence labeling) — это задача, где каждому слову предложения нужно приписать метку: часть речи, тип именованной сущности и т. п. Скрытая марковская модель (HMM) считает, что теги $t_1 \dots t_n$ образуют марковскую цепь, а каждое слово $w_i$ «порождается» своим тегом:
$$P(w, t) = \prod_{i=1}^{n} P(t_i \mid t_{i-1})\, P(w_i \mid t_i) \cdot P(\langle/s\rangle \mid t_n), \qquad t_0 = \langle s\rangle.$$

Обе таблицы оценим по корпусу. Ниже около 40 коротких предложений с тегами UD UPOS: NOUN — существительное (СУЩ), VERB — глагол (ГЛАГ), ADJ — прилагательное (ПРИЛ), ADV — наречие (НАР), PRON — местоимение (МЕСТ), DET — местоименное определение («мой», «этот»), ADP — предлог (ПРЕДЛ), NUM — числительное (ЧИСЛ), PART — частица (ЧАСТ), CCONJ — сочинительный союз (СОЮЗ). В корпус специально включены омонимы: «мыла», «стекло», «печь», «стали», «мой», «три».

In [ ]:
TRAIN_TEXT = """
мама/NOUN мыла/VERB раму/NOUN
мама/NOUN мыла/VERB посуду/NOUN после/ADP ужина/NOUN
на/ADP полке/NOUN лежит/VERB кусок/NOUN мыла/NOUN
дети/NOUN купили/VERB душистого/ADJ мыла/NOUN
стекло/NOUN разбилось/VERB на/ADP три/NUM части/NOUN
молоко/NOUN стекло/VERB со/ADP стола/NOUN
масло/NOUN медленно/ADV стекло/VERB на/ADP пол/NOUN
он/PRON протёр/VERB стекло/NOUN мягкой/ADJ тряпкой/NOUN
новое/ADJ стекло/NOUN ярко/ADV блестело/VERB
в/ADP старом/ADJ доме/NOUN стояла/VERB большая/ADJ печь/NOUN
печь/NOUN быстро/ADV нагрелась/VERB
мы/PRON любим/VERB печь/VERB блины/NOUN
бабушка/NOUN затопила/VERB печь/NOUN
дети/NOUN стали/VERB читать/VERB книги/NOUN
ножи/NOUN из/ADP стали/NOUN не/PART ржавеют/VERB
мой/DET брат/NOUN читает/VERB книгу/NOUN
мой/VERB руки/NOUN перед/ADP едой/NOUN
мой/DET друг/NOUN живёт/VERB в/ADP городе/NOUN
три/VERB пятно/NOUN губкой/NOUN
три/NUM студента/NOUN пришли/VERB в/ADP библиотеку/NOUN
я/PRON купил/VERB три/NUM книги/NOUN
студент/NOUN читает/VERB интересную/ADJ книгу/NOUN в/ADP библиотеке/NOUN
учитель/NOUN дал/VERB ученикам/NOUN новые/ADJ книги/NOUN
она/PRON быстро/ADV написала/VERB письмо/NOUN
мы/PRON живём/VERB в/ADP большом/ADJ доме/NOUN
вечером/ADV мы/PRON пили/VERB чай/NOUN
этот/DET город/NOUN очень/ADV красивый/ADJ
эта/DET книга/NOUN лежит/VERB на/ADP столе/NOUN
кошка/NOUN спит/VERB на/ADP тёплой/ADJ печи/NOUN
брат/NOUN и/CCONJ сестра/NOUN пошли/VERB в/ADP школу/NOUN
он/PRON не/PART любит/VERB холодный/ADJ чай/NOUN
мама/NOUN начала/VERB печь/VERB пирог/NOUN
старый/ADJ учитель/NOUN рассказал/VERB интересную/ADJ историю/NOUN
мы/PRON видели/VERB высокие/ADJ горы/NOUN
ученики/NOUN пишут/VERB диктант/NOUN а/CCONJ учитель/NOUN читает/VERB текст/NOUN
зимой/ADV в/ADP деревне/NOUN очень/ADV тихо/ADV
она/PRON часто/ADV пишет/VERB письма/NOUN подруге/NOUN
отец/NOUN починил/VERB старое/ADJ окно/NOUN
дождь/NOUN шёл/VERB весь/DET день/NOUN
два/NUM друга/NOUN долго/ADV гуляли/VERB по/ADP парку/NOUN
сестра/NOUN читает/VERB книгу/NOUN брата/NOUN
пять/NUM учеников/NOUN решили/VERB задачу/NOUN
"""
TEST_TEXT = """
бабушка/NOUN решила/VERB печь/VERB пироги/NOUN
маленькая/ADJ девочка/NOUN рисовала/VERB зелёную/ADJ траву/NOUN
мой/DET отец/NOUN работает/VERB на/ADP заводе/NOUN
студенты/NOUN внимательно/ADV слушали/VERB лекцию/NOUN
кошка/NOUN тихо/ADV спала/VERB на/ADP окне/NOUN
брат/NOUN купил/VERB новую/ADJ машину/NOUN
мы/PRON долго/ADV искали/VERB старую/ADJ карту/NOUN
учитель/NOUN и/CCONJ ученики/NOUN читали/VERB интересный/ADJ рассказ/NOUN
три/NUM девочки/NOUN пели/VERB весёлую/ADJ песню/NOUN
она/PRON аккуратно/ADV мыла/VERB стекло/NOUN
дети/NOUN играли/VERB за/ADP домом/NOUN
"""

def read_tagged(text):
    # каждая строка — предложение, каждый токен записан как слово/ТЕГ
    return [[tuple(tok.rsplit("/", 1)) for tok in line.split()]
            for line in text.strip().splitlines() if line.strip()]

train = read_tagged(TRAIN_TEXT)
test = read_tagged(TEST_TEXT)
train_vocab = {w for s in train for w, _ in s}
print(f"обучение: {len(train)} предложений, {sum(map(len, train))} токенов")
print(f"тест:    {len(test)} предложений, {sum(map(len, test))} токенов")
print(f"незнакомых слов в тесте: {sum(w not in train_vocab for s in test for w, _ in s)}")
print(train[0])

Оценка максимального правдоподобия (MLE) — это просто доли: $P(t_i \mid t_{i-1}) = \frac{C(t_{i-1}, t_i)}{C(t_{i-1})}$ и $P(w \mid t) = \frac{C(t, w)}{C(t)}$. В маленьком корпусе многие пары не встретились ни разу. MLE даёт им ноль, а один ноль обнуляет всё произведение. Поэтому к каждому счётчику добавляем небольшое $k$ (сглаживание add-k, см. практикум 4): $k = 0{,}1$ для переходов и $k = 0{,}01$ для эмиссий. В словаре эмиссий резервируем одно место под незнакомое слово.

Посмотрим на омонимы: сколько раз каждое слово встретилось с каждым тегом и какой получился $\log P(w \mid t)$.

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
pd.set_option("display.width", 200)

# считаем пары (предыдущий тег, тег), пары (тег, слово) и частоты тегов
START, END = "<s>", "</s>"
trans_count, emit_count, tag_count = Counter(), Counter(), Counter()
for sent in train:
    prev = START
    for w, t in sent:
        trans_count[prev, t] += 1
        emit_count[t, w] += 1
        tag_count[t] += 1
        prev = t
    trans_count[prev, END] += 1

TAGS = sorted(tag_count)
VOCAB = sorted(train_vocab)
prev_states = [START] + TAGS     # строки: откуда
next_states = TAGS + [END]       # столбцы: куда

# сглаживание add-k: переходы и эмиссии
K_TRANS, K_EMIT = 0.1, 0.01
C = np.array([[trans_count[p, t] for t in next_states] for p in prev_states], dtype=float)
A = (C + K_TRANS) / (C.sum(axis=1, keepdims=True) + K_TRANS * len(next_states))
logA = np.log(A)
tag_total = np.array([tag_count[t] for t in TAGS], dtype=float)

def emit_logp(word):
    # для неизвестного слова счётчик 0, остаётся только сглаживание; +1 в словаре = место под незнакомое слово
    counts = np.array([emit_count[t, word] for t in TAGS], dtype=float)
    return np.log((counts + K_EMIT) / (tag_total + K_EMIT * (len(VOCAB) + 1)))

print("Счётчики переходов C(предыдущий → следующий):")
print(pd.DataFrame(C.astype(int), index=prev_states, columns=next_states).to_string())
print("\nВероятности переходов P(следующий | предыдущий), сглаженные:")
print(pd.DataFrame(A, index=prev_states, columns=next_states).round(2).to_string())
print()

print("Омонимы: счётчики C(тег, слово) и log P(слово | тег)")
for w in ["мыла", "стекло", "печь", "стали", "мой", "три"]:
    parts = [f"{t} {emit_count[t, w]} ({emit_logp(w)[j]:.2f})" for j, t in enumerate(TAGS) if emit_count[t, w]]
    print(f"   {w:<8}", " | ".join(parts))

Таблица переходов картинкой. Видно, что после предлога (ADP) почти всегда идёт существительное или прилагательное, а после DET — существительное.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(A, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(next_states)), next_states, rotation=45)
ax.set_yticks(range(len(prev_states)), prev_states)
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        if A[i, j] >= 0.05:
            ax.text(j, i, f"{A[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if A[i, j] > 0.5 else "black")
ax.set_xlabel("следующий тег")
ax.set_ylabel("предыдущий тег")
ax.set_title("Вероятности переходов P(тег | предыдущий тег)")
plt.colorbar(im, fraction=0.04)
plt.tight_layout()
plt.show()

## 2. Алгоритм Витерби: решётка шаг за шагом

Перебрать все последовательности тегов невозможно: их $|T|^n$. Алгоритм Витерби решает задачу динамическим программированием. Пусть $\delta_i(t)$ — логарифм вероятности лучшего пути, который заканчивается тегом $t$ на слове $i$:
$$\delta_i(t) = \max_{t'} \big[\delta_{i-1}(t') + \log P(t \mid t')\big] + \log P(w_i \mid t).$$
Указатель $\psi_i(t)$ запоминает, из какого $t'$ пришёл максимум. В конце добавляем переход в $\langle/s\rangle$ и идём по указателям назад. Считаем в логарифмах, потому что произведение многих маленьких вероятностей быстро становится машинным нулём.

Пример: «мама любит печь пирог». Слово «печь» в корпусе бывает и существительным, и глаголом. Примеры из лаборатории («мама мыла раму», «стекло стекло со стола») разберём в разделе 3.

In [ ]:
def viterbi(words, emit=emit_logp, trans=None, verbose=False):
    L = logA if trans is None else trans
    n, T = len(words), len(TAGS)
    E = np.array([emit(w) for w in words])          # E[i, j] = log P(слово_i | тег_j) — заранее для всех слов
    delta = np.zeros((n, T))
    back = np.zeros((n, T), dtype=int)
    delta[0] = L[0, :T] + E[0]
    for i in range(1, n):
        scores = delta[i - 1][:, None] + L[1:, :T]      # scores[k, j] = δ[i-1, k] + log P(тег_j | тег_k): из какого тега выгоднее прийти в тег j
        back[i] = scores.argmax(axis=0)
        delta[i] = scores.max(axis=0) + E[i]
    final = delta[-1] + L[1:, T]                        # переход в конец предложения </s>
    path = [int(final.argmax())]
    for i in range(n - 1, 0, -1):                       # обратный проход по указателям
        path.append(int(back[i, path[-1]]))
    path.reverse()
    if verbose:
        show_trellis(words, E, delta, back, final, path)
    return [TAGS[j] for j in path], float(final.max()), delta, back

def show_trellis(words, E, delta, back, final, path):
    for i, w in enumerate(words):
        print(f"слово {i + 1}: «{w}»  (показаны 3 лучших тега, остальные хуже)")
        for j in np.argsort(-delta[i])[:3]:
            t = TAGS[j]
            if i == 0:
                print(f"   δ({t:<5}) = log P({t}|<s>) {logA[0, j]:6.2f} + log P({w}|{t}) {E[i, j]:6.2f} = {delta[i, j]:7.2f}")
            else:
                k = back[i, j]
                print(f"   δ({t:<5}) = δ({TAGS[k]}) {delta[i - 1, k]:6.2f} + log P({t}|{TAGS[k]}) {logA[k + 1, j]:6.2f}"
                      f" + log P({w}|{t}) {E[i, j]:6.2f} = {delta[i, j]:7.2f}   ← из {TAGS[k]}")
    print("конец:")
    for j in np.argsort(-final)[:3]:
        print(f"   δ({TAGS[j]:<5}) {delta[-1, j]:6.2f} + log P(</s>|{TAGS[j]}) {logA[j + 1, len(TAGS)]:6.2f} = {final[j]:7.2f}")
    print("лучший путь:", " ".join(f"{w}/{TAGS[j]}" for w, j in zip(words, path)))

demo = "мама любит печь пирог".split()
best_tags, best_score, delta, back = viterbi(demo, verbose=True)
print("log P(слова, теги) =", round(best_score, 2))

Та же решётка картинкой. Рамки и стрелки показывают найденный путь. Обратите внимание на столбец «печь»: в нём NOUN и VERB близки, а решает следующий столбец, потому что переход VERB → NOUN гораздо вероятнее, чем NOUN → NOUN.

In [ ]:
def plot_trellis(words, delta, tags):
    rel = (delta - delta.max(axis=1, keepdims=True)).T   # цвет: насколько клетка хуже лучшей в своём столбце (0 = лучшая)
    fig, ax = plt.subplots(figsize=(1.7 * len(words) + 2, 5))
    ax.imshow(np.maximum(rel, -12), cmap="Blues", vmin=-12, vmax=0, aspect="auto")
    for i in range(len(words)):
        for j in range(len(TAGS)):
            ax.text(i, j, f"{delta[i, j]:.1f}", ha="center", va="center", fontsize=8,
                    color="white" if rel[j, i] > -3 else "black")
    path = [TAGS.index(t) for t in tags]
    for i in range(1, len(words)):
        ax.annotate("", xy=(i - 0.3, path[i]), xytext=(i - 0.7, path[i - 1]),
                    arrowprops=dict(arrowstyle="->", color="crimson", lw=2))
    for i, j in enumerate(path):
        ax.add_patch(plt.Rectangle((i - 0.5, j - 0.5), 1, 1, fill=False, ec="crimson", lw=2))
    ax.set_xticks(range(len(words)), words)
    ax.set_yticks(range(len(TAGS)), TAGS)
    ax.set_title("Решётка Витерби: число — δ, цвет — отставание от лучшей клетки столбца", fontsize=10)
    plt.tight_layout()
    plt.show()

plot_trellis(demo, delta, best_tags)

## 3. Проверка: полный перебор и жадный алгоритм

Полный перебор считает $\log P(w, t)$ для всех $|T|^n$ путей. Для 4 слов и 10 тегов это 10 000 путей: перебрать их ещё можно, и ответ обязан совпасть с Витерби. Жадный алгоритм идёт слева направо, берёт лучший тег для текущего слова с учётом предыдущего тега и больше не пересматривает решение. Он не видит следующих слов, поэтому ошибается там, где неоднозначность снимает правый контекст.

In [ ]:
import itertools

def seq_logp(words, tags, emit=emit_logp):
    idx = [TAGS.index(t) for t in tags]
    lp = logA[0, idx[0]] + emit(words[0])[idx[0]]
    for i in range(1, len(words)):
        lp += logA[idx[i - 1] + 1, idx[i]] + emit(words[i])[idx[i]]
    return lp + logA[idx[-1] + 1, len(TAGS)]

def brute_force(words):
    # полный перебор: |T|^n путей
    E = [emit_logp(w) for w in words]
    best, best_lp, n_paths = None, -np.inf, 0
    for idx in itertools.product(range(len(TAGS)), repeat=len(words)):
        lp = logA[0, idx[0]] + E[0][idx[0]]
        for i in range(1, len(words)):
            lp += logA[idx[i - 1] + 1, idx[i]] + E[i][idx[i]]
        lp += logA[idx[-1] + 1, len(TAGS)]
        n_paths += 1
        if lp > best_lp:
            best, best_lp = [TAGS[j] for j in idx], lp
    return best, best_lp, n_paths

def greedy(words, emit=emit_logp):
    # жадно: лучший тег для текущего слова, решение больше не меняем
    prev, out = 0, []
    for w in words:
        j = int((logA[prev, :len(TAGS)] + emit(w)).argmax())
        out.append(TAGS[j])
        prev = j + 1
    return out

for s in ["мама любит печь пирог", "мама мыла раму", "стекло стекло со стола", "мы любим печь блины", "дети стали читать книги"]:
    words = s.split()
    v_tags, v_lp, _, _ = viterbi(words)
    b_tags, b_lp, n_paths = brute_force(words)
    g_tags = greedy(words)
    print(f"предложение: {s}")
    print(f"   {'Витерби':<9}: {' '.join(v_tags):<28} log P = {v_lp:7.2f}")
    print(f"   {'перебор':<9}: {' '.join(b_tags):<28} log P = {b_lp:7.2f}  ({n_paths} путей перебрано)")
    g_lp = seq_logp(words, g_tags)
    mark = "совпадает с Витерби" if g_tags == v_tags else "ОТЛИЧАЕТСЯ"
    print(f"   {'жадный':<9}: {' '.join(g_tags):<28} log P = {g_lp:7.2f}  {mark}\n")

Почему перебор не годится для реальных предложений: Витерби делает $n \cdot |T|^2$ операций, перебор — $|T|^n$. Для предложения из 15 слов и 10 тегов это 1 500 против $10^{15}$.

In [ ]:
n = np.arange(1, 16)
T = len(TAGS)
plt.figure(figsize=(6, 3.5))
plt.semilogy(n, n * T ** 2, "o-", label="Витерби: n·|T|²")
plt.semilogy(n, float(T) ** n, "s-", label="перебор: |T|ⁿ")
plt.xlabel("длина предложения n")
plt.ylabel("операций (лог. шкала)")
plt.title("Число операций: Витерби против перебора")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Точность на новых предложениях и незнакомые слова

Проверим модель на 11 отложенных предложениях. Слово, которого не было в обучении (OOV, out of vocabulary), получает $P(w \mid t) = \frac{k}{C(t) + k(|V|+1)}$. Эта величина больше у редких тегов (меньше знаменатель), поэтому незнакомые слова часто получают странные теги вроде CCONJ или PART.

In [ ]:
def evaluate(data, emit=emit_logp, trans=None, show_errors=False):
    # точность отдельно на знакомых и незнакомых (OOV) словах
    ok = {"known": 0, "unk": 0}
    total = {"known": 0, "unk": 0}
    errors = []
    for sent in data:
        words = [w for w, _ in sent]
        gold = [t for _, t in sent]
        pred, _, _, _ = viterbi(words, emit=emit, trans=trans)
        for w, g, p in zip(words, gold, pred):
            kind = "known" if w in train_vocab else "unk"
            total[kind] += 1
            ok[kind] += g == p
        if pred != gold:
            errors.append(" ".join(
                f"{w}{'' if w in train_vocab else '*'}/{p}" + ("" if p == g else f"({g})")
                for w, g, p in zip(words, gold, pred)))
    acc = {k: ok[k] / total[k] for k in ok}
    acc["all"] = (ok["known"] + ok["unk"]) / (total["known"] + total["unk"])
    print(f"знакомые: {ok['known']}/{total['known']} = {acc['known']:.0%}   "
          f"незнакомые: {ok['unk']}/{total['unk']} = {acc['unk']:.0%}   всего: {acc['all']:.0%}")
    if show_errors and errors:
        print("Ошибки (* — незнакомое слово, в скобках — правильный тег):")
        for e in errors:
            print("   ", e)
    return acc

acc_plain = evaluate(test, show_errors=True)

Исправим это идеей теггера TnT: часть речи незнакомого слова хорошо угадывается по окончанию («-ла», «-ли» у глаголов прошедшего времени, «-ую», «-ый» у прилагательных, «-но» у наречий). Служебные слова (предлоги, союзы, местоимения) — закрытые классы, почти все они уже есть в словаре, поэтому незнакомому слову разрешаем только открытые теги: NOUN, VERB, ADJ, ADV. По обучающим словам этих классов считаем $P(t \mid \text{суффикс})$ для окончаний длиной 1–3, берём самый длинный суффикс, встретившийся хотя бы дважды, сглаживаем к общей частоте тегов и по формуле Байеса получаем
$$P(w \mid t) = \frac{P(t \mid w)\,P(w)}{P(t)} \propto \frac{P(t \mid \text{суффикс})}{P(t)}.$$
Множитель $P(w)$ одинаков для всех тегов и на выбор пути не влияет.

In [ ]:
from collections import defaultdict

OPEN_TAGS = ["ADJ", "ADV", "NOUN", "VERB"]   # незнакомое слово почти всегда из открытого класса: служебные слова почти все уже есть в словаре
is_open = np.array([t in OPEN_TAGS for t in TAGS])
SUFFIX_MAX, MIN_COUNT = 3, 2
suffix_count = defaultdict(Counter)       # счётчики тегов для окончаний длиной 1..SUFFIX_MAX
for sent in train:
    for w, t in sent:
        if t in OPEN_TAGS:
            for k in range(1, SUFFIX_MAX + 1):
                if len(w) > k:
                    suffix_count[w[-k:]][t] += 1
tag_prior = np.where(is_open, tag_total, 0) / tag_total[is_open].sum()

def suffix_tag_probs(word, m=1.0):
    # самый длинный суффикс, встретившийся хотя бы MIN_COUNT раз; сглаживаем к общей частоте тегов
    for k in range(SUFFIX_MAX, 0, -1):
        c = suffix_count.get(word[-k:])
        if c and sum(c.values()) >= MIN_COUNT:
            counts = np.array([c[t] for t in TAGS], dtype=float)
            return (counts + m * tag_prior) / (counts.sum() + m), word[-k:]
    return tag_prior, ""

def emit_logp_suffix(word):
    if word in train_vocab:
        return emit_logp(word)
    p, _ = suffix_tag_probs(word)
    # Байес: P(w|t) ∝ P(t|суффикс) / P(t); множитель, общий для всех тегов, на путь не влияет
    out = np.full(len(TAGS), -np.inf)
    out[is_open] = np.log(p[is_open] / tag_prior[is_open]) + np.log(1 / len(VOCAB))
    return out

print("P(тег | суффикс) по обучающим словам (3 самых частых тега):")
for suf in ["ла", "ли", "ую", "ый", "но", "ки"]:
    c = suffix_count[suf]
    top = ", ".join(f"{t} {x / sum(c.values()):.2f}" for t, x in c.most_common(3))
    print(f"   -{suf:<4} n={sum(c.values()):<3} {top}")

Сравним две модели. Знакомые слова размечаются одинаково, вся разница — в незнакомых. В конце видно, что суффикс — только подсказка. У «заводе», «окне» отношение $P(t \mid \text{-е}) / P(t)$ больше у ADJ («новое», «старое»), но после предлога Витерби выбирает NOUN. Оставшиеся ошибки — в предложении «дети играли за домом»: предлога «за» не было в обучении, а закрытые теги незнакомым словам мы запретили. Ошибка на «за» тянет за собой и «домом»: после NOUN модель ждёт глагол или конец предложения, а не существительное. Так ошибки в HMM распространяются по цепочке.

In [ ]:
print("HMM без суффиксов:")
acc_plain = evaluate(test)
print("HMM + суффиксы:")
acc_suffix = evaluate(test, emit=emit_logp_suffix, show_errors=True)

print("\nНезнакомые слова теста: тег только по суффиксу и тег, выбранный Витерби с учётом соседей:")
for sent in test:
    words = [w for w, _ in sent]
    pred, _, _, _ = viterbi(words, emit=emit_logp_suffix)
    for w, (_, gold), p_ctx in zip(words, sent, pred):
        if w not in train_vocab:
            p, suf = suffix_tag_probs(w)
            alone = TAGS[int(np.argmax(np.where(is_open, p / np.where(is_open, tag_prior, 1), -1)))]
            mark = "✓" if p_ctx == gold else "✗ " + gold
            print(f"   {w:<14} -{suf:<4} суффикс: {alone:<5} Витерби: {p_ctx:<5} {mark}")

labels = ["знакомые", "незнакомые", "всего"]
x = np.arange(3)
plt.figure(figsize=(6, 3.2))
plt.bar(x - 0.2, [acc_plain[k] for k in ("known", "unk", "all")], 0.4, label="HMM без суффиксов", color="#9AA9C9")
plt.bar(x + 0.2, [acc_suffix[k] for k in ("known", "unk", "all")], 0.4, label="HMM + суффиксы", color="#2B55C9")
plt.xticks(x, labels)
plt.ylim(0, 1.05)
plt.ylabel("доля верных тегов")
plt.title("Точность на тесте")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

## 5. То же с библиотекой: `nltk`

В `nltk` есть готовая HMM: `HiddenMarkovModelTrainer.train_supervised` считает те же частоты и сглаживает их (`LidstoneProbDist` — это то же сглаживание add-k). Отличия два: в модели `nltk` одно $k$ для всех таблиц и нет перехода в конец предложения $\langle/s\rangle$. Если настроить нашу HMM так же, разметка должна совпасть с `nltk` полностью — это хорошая проверка нашего кода. Потом сравним точность.

In [ ]:
from nltk.tag.hmm import HiddenMarkovModelTrainer
from nltk.probability import LidstoneProbDist

# nltk ожидает список предложений из пар (слово, тег) — как наш train
trainer = HiddenMarkovModelTrainer(states=list(TAGS), symbols=list(VOCAB))
nltk_hmm = trainer.train_supervised(train, estimator=lambda fd, bins: LidstoneProbDist(fd, K_EMIT, bins))

# наша HMM в «режиме nltk»: одно k для всех таблиц и без перехода в </s>
T = len(TAGS)
logA_nltk = np.zeros_like(logA)      # последний столбец (</s>) оставляем нулевым: он ни на что не влияет
logA_nltk[:, :T] = np.log((C[:, :T] + K_EMIT) / (C[:, :T].sum(axis=1, keepdims=True) + K_EMIT * T))

def emit_logp_nltk(word):
    counts = np.array([emit_count[t, word] for t in TAGS], dtype=float)
    return np.log((counts + K_EMIT) / (tag_total + K_EMIT * len(VOCAB)))

sents = [[w for w, _ in s] for s in test] + [s.split() for s in ["мама любит печь пирог", "мама мыла раму", "стекло стекло со стола", "мы любим печь блины", "дети стали читать книги"]]
same = sum(viterbi(ws, emit=emit_logp_nltk, trans=logA_nltk)[0] == [t for _, t in nltk_hmm.tag(ws)] for ws in sents)
print(f"предложений, где наша HMM в режиме nltk и nltk дают одинаковые теги: {same}/{len(sents)}\n")

ok = sum(p == g for s in test for p, (_, g) in zip([t for _, t in nltk_hmm.tag([w for w, _ in s])], s))
print(f"nltk HMM          : {ok}/{sum(map(len, test))} = {ok / sum(map(len, test)):.0%}")
print("наша, режим nltk  :", end=" ")
evaluate(test, emit=emit_logp_nltk, trans=logA_nltk)
print("наша, с </s>      :", end=" ")
evaluate(test)

print("\nПримеры из раздела 3, где полная модель и nltk расходятся:")
for s in ["мама любит печь пирог", "мама мыла раму", "стекло стекло со стола", "мы любим печь блины", "дети стали читать книги"]:
    words = s.split()
    ours = viterbi(words)[0]
    theirs = [t for _, t in nltk_hmm.tag(words)]
    if ours != theirs:
        print(f"   {s:<26} наша: {' '.join(ours):<22} nltk: {' '.join(theirs)}")

В «режиме nltk» наша реализация даёт ровно те же теги, значит, код верен. Точность же у полной модели заметно выше. Причина — переход в $\langle/s\rangle$: модель знает, что русское предложение обычно кончается существительным и почти никогда предлогом или союзом. Без него хвост предложения из незнакомых слов размечается почти случайно. В «мама любит печь пирог» `nltk` выбирает «печь» = NOUN: без перехода в $\langle/s\rangle$ строка NOUN нормируется только по переходам внутри предложения, и $P(\text{NOUN} \mid \text{NOUN})$ почти удваивается (0,05 → 0,09). Этого хватает, чтобы путь NOUN NOUN обогнал VERB NOUN. Для языков с глаголом в конце (узбекский, SOV) переход в $\langle/s\rangle$ ещё важнее.

## 6. Задания

1. **Сглаживание.** Попробуйте `K_EMIT` = 0.001, 0.1, 1 и `K_TRANS` = 0.01, 1 (после изменения перезапустите ячейки разделов 1–4). Как меняется точность на знакомых и незнакомых словах? Почему большое `K_EMIT` вредит омонимам?
2. **Алгоритм прямого хода (forward).** Скопируйте `viterbi` и замените максимум на сумму в логарифмах (`np.logaddexp.reduce(..., axis=0)`). Получится $\log P(w)$ — вероятность предложения, просуммированная по всем путям. Проверьте ответ перебором: `np.logaddexp.reduce([seq_logp(words, t) for t in itertools.product(TAGS, repeat=len(words))])`.
3. **Где жадный ошибается.** Придумайте ещё два предложения из знакомых слов (например, со словами «стали», «мой», «три»), на которых жадный алгоритм и Витерби расходятся. Распечатайте решётку `viterbi(..., verbose=True)` и объясните, какой переход перевесил.
4. **Суффиксы.** Измените `SUFFIX_MAX` (1, 2, 4) и `MIN_COUNT` (1, 3). Как меняется точность на незнакомых словах? Какие слова остаются ошибками и почему?
5. **Матрица ошибок.** Для модели с суффиксами постройте таблицу «правильный тег × предсказанный тег» на тестовых предложениях (`pd.crosstab`). Какие пары тегов путаются чаще всего?

*Совет: можно разметить свои 20–30 предложений, загрузить `my_corpus.txt` через панель «Файлы» и подставить его текст вместо `TRAIN_TEXT`.*

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь